In [1]:
# loading libraries
import numpy as np

import matplotlib.pyplot as plt 
import seaborn as sns

import pandas as pd

In [2]:
# import selfbuild module
import coriolis_module
# check out if import worked as expected
print(dir(coriolis_module))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_total_drift', 'calculate_velocities', 'coriolis_acc', 'direction_vector', 'radius_at_latitude', 'rotation_matrix']


In [3]:
# import functions
import coriolis_functions 
# check out if import worked as expected
print(dir(coriolis_functions))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_total_drift', 'calculate_velocities', 'coriolis_acc', 'haversine', 'main', 'np', 'radius_at_latitude', 'rotation_matrix']


In [4]:
# loading dataframes
fdf = pd.read_csv("data/flights_sample_3m.csv")
ldf = pd.read_csv("data/airports.csv")

In [5]:
# counting initial rows of dataset 
print("Rows-count at the start: ", nrows_atstart := len(fdf))

Rows-count at the start:  3000000


In [6]:
# defining time columns
time_columns = ["ARR_TIME", "DEP_TIME", "WHEELS_OFF", "WHEELS_ON", "CRS_ARR_TIME", "CRS_DEP_TIME"]

In [7]:
# defining a helper function
def convert_time(df, time_columns):
    """ Convert integer time columns to HH:MM format """
    for col in time_columns:
        # Handle missing values and convert times
        df[col] = df[col].fillna(0).astype(int).apply(lambda x: f"{x//100:02d}:{x%100:02d}")
        # Adjust for hours == 24
        df[col] = df[col].replace("24:00", "00:00")
    return df

In [8]:
# convert to datetime
fdf["FL_DATE"] = pd.to_datetime(fdf["FL_DATE"])

In [9]:
# applying the time conversion
fdf = convert_time(fdf, time_columns)

In [10]:
# Combine dates and times into datetime
fdf["arr_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["ARR_TIME"])
fdf["dep_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["DEP_TIME"])

fdf["crs_arr_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["CRS_ARR_TIME"])
fdf["crs_dep_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["CRS_DEP_TIME"])

fdf["woff_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_OFF"])
fdf["won_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_ON"])

In [11]:
# dropping original time columns
fdf = fdf.drop(time_columns, axis=1)

In [12]:
# setting up boolean columns
boolean_columns = ["CANCELLED", "DIVERTED"]
fdf[boolean_columns] = fdf[boolean_columns].astype("bool")

In [13]:
# setting up categorical columns
categorical = ["ORIGIN", "DEST", "ORIGIN_CITY", "DEST_CITY", "AIRLINE", "AIRLINE_DOT", "AIRLINE_CODE", "CANCELLATION_CODE"] 
fdf[categorical] = fdf[categorical].astype("category")

In [14]:
# checking unique values in both datasets
fdf_airports = set(fdf["ORIGIN"].unique()).union(set(fdf["DEST"].unique()))
ldf_airports = set(ldf["IATA"].unique())  # "IATA" is the column with the airport acronyms

In [15]:
# finding missing airport codes in ldf
missing_airports = fdf_airports.difference(ldf_airports)

In [16]:
# dropping rows where "ORIGIN" or "DEST" are in missing_airports
fdf = fdf[~fdf["ORIGIN"].isin(missing_airports) & ~fdf["DEST"].isin(missing_airports)]

In [17]:
# merging fdf with ldf to add geographical data for ORIGIN and DEST
fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")
fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

In [18]:
# dropping original origin and destination columns
fdf = fdf.drop(["ORIGIN", "DEST"], axis=1)

In [19]:
# enforcing categories on newly generated columns
fdf[["IATA_ORIGIN", "IATA_DEST"]] = fdf[["IATA_ORIGIN", "IATA_DEST"]].astype("category")

In [20]:
# checking dtypes of columns
print(fdf.dtypes)

FL_DATE                    datetime64[ns]
AIRLINE                          category
AIRLINE_DOT                      category
AIRLINE_CODE                     category
DOT_CODE                            int64
FL_NUMBER                           int64
ORIGIN_CITY                      category
DEST_CITY                        category
DEP_DELAY                         float64
TAXI_OUT                          float64
TAXI_IN                           float64
ARR_DELAY                         float64
CANCELLED                            bool
CANCELLATION_CODE                category
DIVERTED                             bool
CRS_ELAPSED_TIME                  float64
ELAPSED_TIME                      float64
AIR_TIME                          float64
DISTANCE                          float64
DELAY_DUE_CARRIER                 float64
DELAY_DUE_WEATHER                 float64
DELAY_DUE_NAS                     float64
DELAY_DUE_SECURITY                float64
DELAY_DUE_LATE_AIRCRAFT           

In [21]:
# taking only the not cancelled flights
fdf = fdf[~fdf["CANCELLED"] & (fdf["AIR_TIME"] > 0)]

In [22]:
# calculating haversinedistance
fdf["haversine_distance"] = coriolis_functions.haversine(
    fdf["LATITUDE_ORIGIN"],
    fdf["LONGITUDE_ORIGIN"],
    fdf["LATITUDE_DEST"], 
    fdf["LONGITUDE_DEST"]
)

In [23]:
# Apply direction vector function row-wise
directions = fdf.apply(
    lambda row: coriolis_module.direction_vector(
        row["LATITUDE_ORIGIN"], 
        row["LONGITUDE_ORIGIN"], 
        row["LATITUDE_DEST"], 
        row["LONGITUDE_DEST"]
    ), axis=1
)

In [24]:
# Split the resulting direction vectors into separate columns
fdf["x_direction"] = directions.apply(lambda x: x[0])
fdf["y_direction"] = directions.apply(lambda x: x[1])
fdf["z_direction"] = directions.apply(lambda x: x[2])

In [ ]:
fdf["total_drift"] = fdf.apply(lambda row: coriolis_module.calculate_total_drift(row[["haversine_distance", "x_direction", "y_direction", "z_direction", "LATITUDE_ORIGIN", "LATITUDE_DEST"]], row["AIR_TIME"]), axis=1)
